# Single-value calibration of an acoustic measurement chain

*Simon Kersten (1), Marco Berzborn (2)*<br>  
*(1) Institute for Hearing Technology and Acoustics, RWTH Aachen University*<br>  
*(2) Department of the Built Environment, Eindhoven University of Technology*<br> 

*Contact: simon.kersten@akustik.rwth-aachen.de, m.berzborn@tue.nl*

In this tutorial, you will determine and verify the conversion factors of an acoustic measurement chain.
A typical acoustic measurement chain, as for example used for measuring a loudspeaker transfer function, consists of:

- **Input chain:** microphone → input amplifier → analog-to-digital converter (ADC) → digital samples
- **Output chain:** digital samples → digital-to-analog converter (DAC) → output amplifier → electrical output

The following steps cover the calibration of the input chain (ADC and microphone combined and individually) and DAC.
Optional extensions include the calibration of input- and output-amplifier gains.

The calibration is performed using an acoustic calibrator producing a known sound pressure level, for example 94 dB SPL at 1 kHz (1 Pa RMS), and by manually measuring the output voltage of the DAC. You will use these quantities to calculate the corresponding conversion factors, defined as *output quantity / input quantity*:

| Conversion factor | Definition | Unit |
|---|---|---|
| Input chain | Digital sample value / sound pressure | 1/Pa |
| DAC | Output voltage / digital sample value | V/1 |
| ADC | Digital sample value / input voltage | 1/V |
| Microphone sensitivity | Microphone output voltage / sound pressure | V/Pa |



**Duration:** 90 Minutes

**Requirements:**
- Basic understanding of sine signals, RMS values, voltage, and sound pressure level
- An audio interface with at least one input and one output
- A microphone and a compatible acoustic calibrator
- A true-RMS voltmeter
- A loopback connection between input and output


**Dependencies**<br>  
`pip install pyfar>=0.8 pooch nbgrader watermark sounddevice ipympl matplotlib`

## Configuration

Before starting the measurements, configure the measurement parameters and select the audio interface. The examples below use **CoreAudio on macOS** as the default configuration. On Windows, you can use an ASIO or WASAPI device instead. The available audio interfaces and their corresponding host APIs depend on your operating system and installed audio drivers.

**Adapt these settings to match your hardware.**

In [ ]:
import os

# Windows: Enable ASIO support before importing sounddevice.
# Uncomment this line if you intend to use an ASIO device:
# os.environ["SD_ENABLE_ASIO"] = "1"

import pyfar as pf
import numpy as np
import matplotlib.pyplot as plt
import sounddevice as sd

%matplotlib ipympl

In [ ]:
# Audio settings
sampling_rate = 44100       # Sampling rate in Hz
# List available audio interfaces and their host APIs
sd.query_devices()

In [ ]:
# Select the audio interface.
# Replace the example with the exact device name shown by sd.query_devices().
sd.default.device = "Fireface UCX"

# Default audio settings: (number of input channels, number of output channels)
sd.default.samplerate = sampling_rate
sd.default.channels = (1, 1)

# Physical channel numbers (sounddevice mappings are one-based)
input_channel = 1
output_channel = 1

# Optional macOS CoreAudio settings.
# Use these to request the device to change its sampling rate rather than
# silently resampling, and raise an error if conversion would be required.
#
# sd.default.extra_settings = sd.CoreAudioSettings(
#     change_device_parameters=True,
#     fail_if_conversion_required=True)

print("Selected device:", sd.default.device)
print("Sampling rate:", sd.default.samplerate)
print("Default channel counts:", sd.default.channels)
print("Input channel mapping:", [input_channel])
print("Output channel mapping:", [output_channel])

The following helper function records one input channel and returns a `pyfar.Signal`.

In [ ]:
def record_signal(duration):
    """Record one input channel and return the recording as a pyfar.Signal."""
    n_samples = int(round(duration * sampling_rate))
    recording = sd.rec(
        frames=n_samples,
        samplerate=sampling_rate,
        mapping=[input_channel],
        blocking=True
    )
    recording = np.asarray(recording).squeeze()
    recorded_signal = pf.Signal(
        recording, sampling_rate=sampling_rate, fft_norm='rms')
    return recorded_signal

## Input-chain calibration

**Objective:** Determine the input-chain conversion factor from microphone sound pressure to digital samples.

**Reference:** Acoustic calibrator (e.g. pistonphone) producing a known sound pressure level at a specified frequency, for example 94 dB SPL at 1 kHz (1 Pa RMS).

### Step 1: Mount the microphone and calibrator

Mount the microphone securely and ensure a proper acoustic seal.

### Step 2: Record and inspect

Record the calibration signal and inspect its waveform and spectrum. Check:
- Expected reference frequency
- Stable amplitude over time
- No clipping
- Sufficient signal-to-noise ratio

Repeat the measurement if necessary.

In [ ]:
measurement_duration = 10  # Recording duration in seconds
# Record
input_recording = record_signal(measurement_duration)
# Inspect
fig = plt.figure()
ax = pf.plot.time_freq(input_recording)
ax[0].set_title("Input chain recording")
plt.show()

### Step 3: Noise removal
Apply a high-pass filter to remove the noise below 500 Hz. Discuss the effect of the filtering in time and frequency domain.

In [ ]:

input_recording_highpass = pf.dsp.filter.butterworth(
    input_recording, 8, frequency=500, btype='highpass')

fig = plt.figure()
ax = pf.plot.time_freq(input_recording, label="Raw recording")
pf.plot.time_freq(input_recording_highpass, label="High-pass filter applied")
ax[1].legend()
plt.show()

### Step 4: Calculate the input-chain conversion factor

- Determine the digital RMS value using [`pyfar.dsp.rms`](https://pyfar.readthedocs.io/en/stable/modules/pyfar.dsp.html#pyfar.dsp.rms).
- Calculate the input chain conversion factor using the nominal sound pressure level of the reference signal.
- Compare the results with and without high-pass filter applied.
- How can we evaluate whether the calculated factor is plausible?

In [ ]:
# Nominal acoustic reference level
reference_level = 94.0  # dB SPL
# Convert SPL to RMS sound pressure
reference_pressure = 20e-6 * 10**(reference_level / 20)  # Pa RMS

# YOUR CODE HERE
raise NotImplementedError()

print(f"Input chain conversion factor, without high-pass: {np.squeeze(input_factor):.6g} 1/Pa")
print(f"Input chain conversion factor, with high-pass: {np.squeeze(input_factor_highpass):.6g} 1/Pa")

## DAC Calibration
**Objective:** Determine the DAC conversion factor from digital samples to output voltage.

**Reference:** A digital signal with a defined RMS value, for example sine signal at 1 kHz.

### Step 1: Generate a reference signal

Generate a 1 kHz sine signal with a defined digital RMS value. Set at an appropriate amplitude.
You can use [``pyfar.signals.sine``](https://pyfar.readthedocs.io/en/stable/modules/pyfar.signals.html#pyfar.signals.sine) and [`pyfar.dsp.rms`](https://pyfar.readthedocs.io/en/stable/modules/pyfar.dsp.html#pyfar.dsp.rms).

Note: Depending on the meter used for measuring the output voltage, you may need to adjust the frequency. Standard voltmeters have a limited frequency range due to sampling rates much lower than audio sampling rates. Consider using a frequency of 100 Hz instead of 1 kHz to avoid aliasing or frequency response roll-off.

In [ ]:
sine_frequency = .1e3 # Reference frequency in Hz
sine_amplitude = 0.5 # Digital amplitude
sine_duration = 30 # in seconds
sine_n_samples = int(round(sine_duration * sampling_rate))

# YOUR CODE HERE
raise NotImplementedError()

fig = plt.figure()
ax = pf.plot.time_freq(sine)
ax[1].set_ylim(-40, 0)
plt.show()

print(f"Digital RMS value: {np.squeeze(output_rms):.6g}")

### Step 2: Measure the RMS voltage

Play back the sine signal and measure the RMS voltage using the voltmeter. Note down the measured value below.

In [ ]:
# Playback of sine signal
sd.play(
    sine.time.T,
    samplerate=sampling_rate,
    mapping=[output_channel],
    blocking=True)

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

print(f"Digital RMS value: {np.squeeze(output_rms):.6g}")
print(f"Measured RMS voltage: {np.squeeze(dac_voltage):.6g}")

### Step 3: Calculate the DAC conversion factor
Calculate the DAC conversion factor using the digital and voltage RMS values.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

print(f"DAC conversion factor: {np.squeeze(dac_factor):.5g} V/1")

**Note:** As an alternative to the input-chain calibration already conducted, the DAC calibration allows us to determine the ADC conversion factor and microphone sensitivity separately.

## ADC calibration via electrical loopback

**Objective:** Determine the ADC conversion factor from input voltage to digital samples. Use the DAC conversion factor to control the loopback voltage digitally.

**Reference:** A digital signal with a defined RMS value, for example sine signal at 1 kHz.

**Alternative:** Directly apply a known ADC input voltage using a suitable reference source.

### Step 1: Connect the electrical loopback

Connect the selected audio output to the selected audio input using a suitable line-level connection.

**Important**: Do not connect a line output to a microphone input with phantom power enabled unless the equipment and connection explicitly support it.

### Step 2: Generate a reference signal

Generate a 1 kHz sine signal with a defined digital RMS value. Add silence after the signal to accommodate the electrical delay. Start with a conservative digital output level and check the input level to avoid clipping.
You can use [``pyfar.signals.sine``](https://pyfar.readthedocs.io/en/stable/modules/pyfar.signals.html#pyfar.signals.sine) and [``pyfar.dsp.pad_zeros``](https://pyfar.readthedocs.io/en/stable/modules/pyfar.dsp.html#pyfar.dsp.pad_zeros) to create the signal.

In [ ]:
sine_frequency = .1e3 # Reference frequency in Hz
sine_level = -40 # Conservative level in dBFS
sine_amplitude = 10**(sine_level/20) # Digital amplitude
sine_duration = 10 # in seconds
sine_n_samples = int(round(sine_duration * sampling_rate))
pad_width = int(round(0.1 * sampling_rate))  # 100 ms padded

# YOUR CODE HERE
raise NotImplementedError()

fig = plt.figure()
ax = pf.plot.time_freq(sine)
plt.show()

print(f"Digital RMS value: {np.squeeze(output_rms):.6g}")

### Step 3: Play and record the signal

Play the reference signal while recording the loopback input.

Inspect the played and recorded signals. Check:
- The expected frequency
- A stable recorded amplitude during the active signal
- No clipping
- Sufficient signal-to-noise ratio

Repeat the measurement with a lower output level if clipping occurs.

In [ ]:
# Record
recording= sd.playrec(
    np.asarray(sine_padded.time).squeeze(),
    samplerate=sampling_rate,
    input_mapping=[input_channel],
    output_mapping=[output_channel],
    blocking=True
)
adc_recording = pf.Signal(
    np.asarray(recording).squeeze(), sampling_rate=sampling_rate,
    fft_norm='rms')

# Inspect
plt.figure()
ax = pf.plot.time_freq(sine_padded, label="Played digital signal")
pf.plot.time_freq(adc_recording, label="Recorded loopback signal")
ax[1].legend()
ax[0].set_title("ADC loopback recording")
plt.show()

### Step 4: Extract signal and calculate the ADC conversion factor

The recorded signal may be delayed relative to the played signal and contains the padded zeros. To calculate the RMS value of the recorded sine signal extract the part of the recording using [`pyfar.dsp.time_crop`](https://pyfar.readthedocs.io/en/stable/modules/pyfar.dsp.html#pyfar.dsp.time_crop). Use this extracted signal to determine the RMS value of the recorded voltage in digital units and calculate the ADC conversion factor.
How much is the conversion factor affected by the length of the segment?

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

fig = plt.figure()
ax = pf.plot.time_freq(adc_recording, label="Recording")
pf.plot.time_freq(adc_recording_segment, label="Extracted segment")
ax[1].legend()
plt.show()

In [ ]:
# Calculate ADC conversion factor
# YOUR CODE HERE
raise NotImplementedError()

print(f"Output digital RMS value: {np.squeeze(output_rms):.6g}")
print(f"Output RMS voltage: {np.squeeze(output_voltage):.6g} V")
print(f"Recorded digital RMS value: {np.squeeze(recorded_rms):.6g}")
print(f"DAC conversion factor: {np.squeeze(dac_factor):.6g} V/1")
print(f"ADC conversion factor: {np.squeeze(adc_factor):.6g} 1/V")

## Microphone sensitivity calibration

**Objective:** Determine the microphone sensitivity in volts per pascal. Use the ADC conversion factor determined in the previous section to convert digital samples to voltage.

**Reference:** Acoustic calibrator (e.g. pistonphone) producing a known sound pressure level at a specified frequency, for example 94 dB SPL at 1 kHz (1 Pa RMS).

**Note:** The procedure resembles the input-chain calibration from above.

### Step 1: Mount the microphone and calibrator

Mount the microphone securely and ensure a proper acoustic seal.

### Step 2: Record and inspect

Record the calibration signal and inspect its waveform and spectrum. Check:
- Expected reference frequency
- Stable amplitude over time
- No clipping
- Sufficient signal-to-noise ratio

Repeat the measurement if necessary.

In [ ]:
measurement_duration = 10  # Recording duration in seconds
# Record
mic_recording = record_signal(measurement_duration)
# Inspect
fig = plt.figure()
ax = pf.plot.time_freq(mic_recording)
ax[0].set_title("Microphone sensitivity calibration recording")
plt.show()

### Step 3: Noise removal
Apply a high-pass filter to remove the noise below 500 Hz.

In [ ]:

mic_recording_highpass = pf.dsp.filter.butterworth(
    mic_recording, 8, frequency=500, btype='highpass')

fig = plt.figure()
ax = pf.plot.time_freq(mic_recording, label="Raw recording")
pf.plot.time_freq(mic_recording_highpass, label="High-pass filter applied")
ax[1].legend()
plt.show()

### Step 4: Calculate the microphone sensitivity

- Determine the digital RMS value.
- Calculate the microphone sensitivity using the nominal sound pressure level of the reference signal and the already determined ADC conversion factor.
- Is the directly measured input-chain conversion factor consistent with the individually measured ADC conversion factor and microphone sensitivity? Interpret differences in relation to the expected measurement uncertainty and repeatability.

In [ ]:
# Calculate the microphone sensitivity and combined input chain conversion factor
# YOUR CODE HERE
raise NotImplementedError()

print(f"Microphone sensitivity: {np.squeeze(mic_sensitivity*1e3):.6g} mV/Pa")
print(f"Measured input-chain factor: {np.squeeze(input_factor):.6g} 1/Pa")
print(f"Combined individual factors: {np.squeeze(input_factor_combined):.6g} 1/Pa")

## Additional notes

### Calibration management

Calibration factors apply to a particular measurement configuration. Store them together with the information needed to reproduce that configuration. Record at least:
- Audio-interface manufacturer and model
- Device name and host API
- Input and output channels
- Sampling rate
- Input and output gain settings
- Microphone and calibrator identification
- Reference values and correction values
- Calibration date and time
- Determined conversion factors

Preserve the previous accepted calibration until the new calibration has been checked and explicitly accepted. Recalibration may be necessary after:
- Replacing the audio interface
- Changing channels or signal paths
- Changing relevant input or output gain settings
- Replacing the microphone
- Changing other hardware that affects the calibrated path

Where appropriate, automate checks of signal level, clipping, reference frequency, and calibration-factor validity.

### Optional extension: Input-amplifier calibration

**Objective:** Determine the gain of an input amplifier in V/V.

**Prerequisites:** Valid ADC and DAC calibration; Access to the amplifier input and output.

**Procedure:** Apply a known reference signal from the DAC to the amplifier input. Measure the amplifier output using the calibrated ADC. Calculate the gain.

### Optional extension: Output-amplifier calibration

**Objective:** Determine the gain of an output amplifier in V/V.

**Prerequisites:** Valid ADC and DAC calibration; Access to the amplifier input and output; Suitable electrical load and measurement connection.

**Procedure:** Apply a known digital signal to the DAC and output amplifier. Measure the amplifier output voltage using the calibrated ADC. Determine the voltage at the amplifier input from the digital signal and DAC conversion factor. Calculate the gain.

# License notice

This notebook © **2026** by **Simon Kersten and Marco Berzborn** is licensed under [CC BY 4.0](http://creativecommons.org/licenses/by/4.0/?ref=chooser-v1)

# Watermark

The following watermark might help others to install specific package versions that might be required to run the notebook. Please give at least the versions of Python, IPython, numpy , and scipy, major third party packagers (e.g., pytorch), and all used pyfar packages.

In [ ]:
%load_ext watermark
%watermark -v -m -p numpy,scipy,matplotlib,pyfar,sounddevice,nbgrader,watermark